# Model Versioning — Examples

This notebook shows a local, reproducible model-versioning workflow using a synthetic dataset. It saves artifacts only in a temporary directory created for this notebook; it does not upload them anywhere.

Install dependencies if needed:

```bash
pip install numpy pandas scikit-learn joblib
```

In [ ]:
import json, hashlib, tempfile, platform
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import sklearn
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import joblib

SEED = 42
ROOT = Path(tempfile.mkdtemp(prefix="model-versioning-"))
ARTIFACT_PATH = ROOT / "risk_model_1.0.0.joblib"
MANIFEST_PATH = ROOT / "model_manifest.json"
print("Working directory:", ROOT)
print("scikit-learn:", sklearn.__version__)

## 1. Create and split a synthetic dataset
The test set remains held out until final evaluation. Preprocessing is placed inside the pipeline to avoid fitting transformations on test data.

In [ ]:
X_array, y = make_classification(
    n_samples=1200, n_features=8, n_informative=5, n_redundant=1,
    weights=[0.7, 0.3], class_sep=0.9, random_state=SEED
)
feature_names = [f"feature_{i}" for i in range(X_array.shape[1])]
X = pd.DataFrame(X_array, columns=feature_names)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED
)
print("Train:", X_train.shape, "Test:", X_test.shape)
print("Train positive rate:", round(float(np.mean(y_train)), 3))
print("Test positive rate:", round(float(np.mean(y_test)), 3))

## 2. Train a candidate model and evaluate it

In [ ]:
model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=1000, random_state=SEED)),
])
model.fit(X_train, y_train)
predictions = model.predict(X_test)

metrics = {
    "accuracy": float(accuracy_score(y_test, predictions)),
    "precision": float(precision_score(y_test, predictions, zero_division=0)),
    "recall": float(recall_score(y_test, predictions, zero_division=0)),
    "f1": float(f1_score(y_test, predictions, zero_division=0)),
}
metrics

## 3. Save the trusted artifact and calculate its digest
Only load artifacts from a trusted source. Joblib uses Python serialization and is not safe for untrusted files.

In [ ]:
joblib.dump(model, ARTIFACT_PATH)

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

artifact_digest = sha256_file(ARTIFACT_PATH)
print("Artifact:", ARTIFACT_PATH.name)
print("SHA-256:", artifact_digest)

## 4. Create a model manifest
The manifest links the model artifact to its data reference, feature schema, code revision, dependencies, and evaluation metrics. Replace illustrative placeholders with real lineage values in a production project.

In [ ]:
manifest = {
    "model_name": "synthetic-risk-classifier",
    "model_version": "1.0.0",
    "artifact_file": ARTIFACT_PATH.name,
    "artifact_sha256": artifact_digest,
    "code_commit": "replace-with-real-git-commit",
    "data_version": "synthetic-dataset-seed-42",
    "feature_schema_version": "features-v1",
    "training_run_id": "local-demo-run-001",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "python_version": platform.python_version(),
    "scikit_learn_version": sklearn.__version__,
    "metrics": metrics,
    "approval_status": "candidate",
}
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(MANIFEST_PATH.read_text(encoding="utf-8"))

## 5. Validate manifest and artifact integrity

In [ ]:
def validate_manifest(manifest: dict) -> None:
    required = [
        "model_name", "model_version", "artifact_file", "artifact_sha256",
        "data_version", "feature_schema_version", "metrics", "approval_status"
    ]
    missing = [key for key in required if not manifest.get(key)]
    if missing:
        raise ValueError(f"Missing required metadata: {missing}")
    if manifest["approval_status"] not in {"candidate", "staging", "production", "archived"}:
        raise ValueError("Unsupported approval_status")
    for metric_name, value in manifest["metrics"].items():
        if not isinstance(value, (int, float)) or not np.isfinite(value):
            raise ValueError(f"Invalid metric: {metric_name}")
        if not 0 <= value <= 1:
            raise ValueError(f"Expected metric in [0, 1]: {metric_name}")

validate_manifest(manifest)
assert sha256_file(ARTIFACT_PATH) == manifest["artifact_sha256"]
print("Manifest valid; artifact digest matches.")

## 6. Reload and verify prediction consistency
This checks that the saved artifact behaves like the in-memory pipeline in the current environment.

In [ ]:
# Safety note: this artifact was created by this notebook in the current temporary directory.
reloaded_model = joblib.load(ARTIFACT_PATH)
sample = X_test.iloc[:10]
original = model.predict(sample)
reloaded = reloaded_model.predict(sample)
assert np.array_equal(original, reloaded)
print("Reloaded predictions match:", bool(np.array_equal(original, reloaded)))

## 7. Apply a simple candidate promotion gate
In real projects, choose thresholds before evaluation and validate on representative data. These thresholds are an example policy only.

In [ ]:
QUALITY_GATE = {
    "minimum_recall": 0.70,
    "minimum_f1": 0.65,
}
gate_passed = (
    metrics["recall"] >= QUALITY_GATE["minimum_recall"]
    and metrics["f1"] >= QUALITY_GATE["minimum_f1"]
)
manifest["approval_status"] = "staging" if gate_passed else "candidate"
manifest["quality_gate"] = QUALITY_GATE
manifest["quality_gate_passed"] = bool(gate_passed)
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print("Gate passed:", gate_passed)
print("Current state:", manifest["approval_status"])

## 8. Compare versions using a common protocol
The next table is illustrative structure for comparing candidate versions. For a fair comparison, use the same held-out evaluation data and metric definitions.

In [ ]:
comparison = pd.DataFrame([
    {"model_version": "1.0.0", **metrics, "artifact_sha256": artifact_digest[:12]},
    # Add another real candidate's measured metrics and digest here.
])
comparison

## 9. Suggested production release record

A production release should link:
- immutable model artifact digest and registry version,
- code commit and training run ID,
- data snapshot and feature schema,
- dependency lock file and serving image digest,
- evaluation evidence and approval,
- deployment timestamp, monitor owner, and rollback target.

Do not store secrets or raw personal data in the manifest. Do not treat a digest as proof of provenance unless the expected digest is obtained from a trusted source.